# Sensitivity Reconciliation

**Objective:** Reconcile the aggregate GMSL sensitivity to GMST (from observations
and the Bayesian rate-and-state model) with the sum of individual component
sensitivities. Identify and close the deficit by adjusting the two GMST-sensitive,
poorly-constrained components: thermosteric expansion and Greenland SMB.

**Rationale:** The satellite-era quadratic extrapolation of GMSL gives
approximately 660 mm at 2100. Our component sum gives 756--1111 mm depending on
SSP, but removing WAIS (390 mm median) leaves 366--721 mm --- below or comparable
to the observational baseline. The non-WAIS physics-based components
underestimate the trends already visible in GMSL. This notebook quantifies the
gap and uses observations to constrain the uncertain components.

**Key assumption:** During the satellite era, WAIS contributes negligibly to GMSL.
Other components --- glaciers, Greenland discharge, EAIS, Antarctic Peninsula ---
are either well-constrained by data or have negligible GMST sensitivity. The
deficit between observed and modeled GMSL therefore resides in thermosteric
expansion and Greenland SMB, both of which are sensitive to GMST but not
well-constrained independently.

In [1]:
import json
import sys
import os

import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-poster')

sys.path.insert(0, '.')
from component_io import (
    load_all_projections, load_component, PROJ_SSPS, PROJ_YEARS,
    DEFAULT_H5_PATH, N_SAMPLES,
)
from bayesian_dols import fit_satellite_era_quadratic

# ── Paths ──
H5_PATH = '../data/processed/slr_processed_data.h5'
H5_COMP = str(DEFAULT_H5_PATH)
RS_JSON = '../data/processed/bayesian_ratestate_results.json'
RS_POSTERIOR_H5 = '../data/processed/bayesian_ratestate_posterior.h5'
S1_JSON = '../data/processed/stage1_thermosteric_results.json'
GR_JSON = '../data/processed/greenland_ct_estimate.json'
ABLAIN_COV = '../data/raw/gmslr/ablain2019_gmsl_error_covariance.nc'
FIG_DIR = '../figures'

# ── Constants ──
BASELINE_YEAR = 2005.0
M_TO_MM = 1000.0
GT_TO_MM_SLE = 1.0 / 362.5  # Gt -> mm SLE

# Hamlington et al. (2024) satellite-era quadratic (reference values)
HAMLINGTON_RATE_MM_YR = 4.5      # mm/yr at end of record (2025)
HAMLINGTON_RATE_RANGE = (3.5, 5.5)
HAMLINGTON_ACCEL_MM_YR2 = 0.08   # mm/yr^2
HAMLINGTON_ACCEL_RANGE = (0.02, 0.14)

# ── Load core datasets ──
# Note: harmonized data is in meters, rebased to 2005 baseline
df_berkeley = pd.read_hdf(H5_PATH, key='harmonized/df_berkeley_h')
df_frederikse = pd.read_hdf(H5_PATH, 'harmonized/df_frederikse_h')

print(f'Berkeley Earth: {len(df_berkeley)} obs, columns: {list(df_berkeley.columns)}')
print(f'Frederikse: {len(df_frederikse)} obs, TWS columns: '
      f'{[c for c in df_frederikse.columns if "tws" in c.lower()]}')

Berkeley Earth: 2100 obs, columns: ['temperature', 'temperature_unc', 'temperature_sigma']
Frederikse: 119 obs, TWS columns: ['tws_lower', 'tws', 'tws_upper', 'tws_natural_lower', 'tws_natural', 'tws_natural_upper', 'tws_sigma', 'tws_natural_sigma']


---
## 1. Aggregate GMSL constraints

Two independent constraints on the aggregate GMSL--GMST relationship:

1. **Satellite-era quadratic fit:** Rate and acceleration from NASA altimetry
   (model-free, no GMST dependence assumed)
2. **Bayesian rate-and-state model:** Explicit GMST sensitivity calibrated
   over 1900--2018

In [2]:
# ── 1a. Satellite-era quadratic (Hamlington et al. 2024) ──
# Reference values from Hamlington et al. (2024):
#   rate = 4.5 mm/yr (3.5--5.5) at end of record
#   accel = 0.08 mm/yr^2 (0.02--0.14)
# We recompute using the Hamlington methodology to obtain the full
# coefficient covariance matrix (not provided in the publication).

# NASA GMSL (harmonized, already in meters)
df_nasa_h = pd.read_hdf(H5_PATH, 'harmonized/df_nasa_gmsl_h')
nasa_time = df_nasa_h['decimal_year'].values
nasa_gmsl = df_nasa_h['gmsl'].values  # meters
nasa_sigma = df_nasa_h['gmsl_sigma'].values if 'gmsl_sigma' in df_nasa_h.columns else None

sq = fit_satellite_era_quadratic(
    nasa_time, nasa_gmsl,
    sigma=nasa_sigma,
    meas_cov_path=ABLAIN_COV if os.path.exists(ABLAIN_COV) else None,
)

print('Satellite-era quadratic fit (Hamlington et al. 2024 methodology)')
print(f'  Window: {sq.t_start:.1f} -- {sq.t_end:.1f} ({sq.n_obs} obs)')
print(f'  Rate at {sq.eval_time:.1f}: {sq.rate * M_TO_MM:.2f} +/- {sq.rate_se * M_TO_MM:.2f} mm/yr')
print(f'    Hamlington reference:  {HAMLINGTON_RATE_MM_YR} ({HAMLINGTON_RATE_RANGE[0]}--{HAMLINGTON_RATE_RANGE[1]}) mm/yr')
print(f'  Acceleration: {sq.accel * 1e6:.1f} +/- {sq.accel_se * 1e6:.1f} um/yr^2')
print(f'    Hamlington reference:  {HAMLINGTON_ACCEL_MM_YR2 * 1e3:.0f} ({HAMLINGTON_ACCEL_RANGE[0] * 1e3:.0f}--{HAMLINGTON_ACCEL_RANGE[1] * 1e3:.0f}) um/yr^2')

# ── 1b. Rate-and-state Bayesian model ──
with open(RS_JSON) as f:
    rs = json.load(f)

rs_cal = rs['calibration']
rs_coeff = rs_cal['coefficients_mm_yr']
rs_hdi = rs_cal['hdi_94_mm_yr']

print('\nRate-and-state model (1900--2018)')
print(f'  R^2 = {rs_cal["r_squared"]:.4f}')
for k in ['dalpha_dT', 'alpha0', 'trend', 'd_diseq']:
    print(f'  {k:12s} = {rs_coeff[k]:+.3f} mm/yr  [{rs_hdi[k][0]:.3f}, {rs_hdi[k][1]:.3f}]')
print(f'  tau          = {rs_cal["tau_yr"]["median"]:.1f} yr  [{rs_cal["tau_yr"]["hdi_94"][0]:.1f}, {rs_cal["tau_yr"]["hdi_94"][1]:.1f}]')

Satellite-era quadratic fit (Hamlington et al. 2024 methodology)
  Window: 1993.0 -- 2025.3 (1191 obs)
  Rate at 2025.3: 4.49 +/- 0.61 mm/yr
    Hamlington reference:  4.5 (3.5--5.5) mm/yr
  Acceleration: 72.8 +/- 36.8 um/yr^2
    Hamlington reference:  80 (20--140) um/yr^2

Rate-and-state model (1900--2018)
  R^2 = 0.9679
  dalpha_dT    = +4.592 mm/yr  [3.115, 6.104]
  alpha0       = +5.209 mm/yr  [4.069, 6.360]
  trend        = +2.670 mm/yr  [2.378, 3.021]
  d_diseq      = +0.695 mm/yr  [0.000, 1.910]
  tau          = 15.0 yr  [3.9, 75.4]


In [3]:
# ── 1c. Empirical drate/dGMST during satellite era ──
# Use the quadratic acceleration and concurrent GMST change
# to estimate how much of the rate increase is attributable to GMST.

# GMST during satellite era
t_sat_start, t_sat_end = sq.t_start, sq.t_end
mask_sat = (df_berkeley.index >= pd.Timestamp(f'{int(t_sat_start)}-01-01')) & \
           (df_berkeley.index <= pd.Timestamp(f'{int(t_sat_end)}-12-31'))
T_sat = df_berkeley.loc[mask_sat, 'temperature']

# Annual means for cleaner trend
T_sat_annual = T_sat.resample('YS').mean().dropna()
T_sat_years = T_sat_annual.index.year.values.astype(float) + 0.5
T_sat_vals = T_sat_annual.values

# Linear trend in GMST during satellite era
p_T = np.polyfit(T_sat_years - t_sat_start, T_sat_vals, 1)
dT_dt = p_T[0]  # deg C / yr
T_start = np.polyval(p_T, 0)
T_end = np.polyval(p_T, t_sat_end - t_sat_start)
Delta_T = T_end - T_start

# Rate change from quadratic acceleration
Delta_rate = sq.accel * M_TO_MM * (t_sat_end - t_sat_start)  # mm/yr

# Empirical sensitivity
drate_dGMST_obs = Delta_rate / Delta_T if abs(Delta_T) > 0.01 else np.nan

print(f'Satellite era ({t_sat_start:.0f}--{t_sat_end:.1f}):')
print(f'  GMST trend:    {dT_dt:.4f} deg C/yr')
print(f'  GMST change:   {Delta_T:.2f} deg C  ({T_start:.2f} -> {T_end:.2f})')
print(f'  Rate change:   {Delta_rate:.2f} mm/yr')
print(f'  Empirical drate/dGMST = {drate_dGMST_obs:.2f} mm/yr per deg C')
print()
print('NOTE: This empirical estimate conflates T-driven and t-driven')
print('acceleration. It is an upper bound on the true GMST sensitivity')
print('if any acceleration is due to non-T processes (e.g., ice-sheet dynamics).')

Satellite era (1993--2025.3):
  GMST trend:    0.0238 deg C/yr
  GMST change:   0.77 deg C  (-0.21 -> 0.55)
  Rate change:   2.36 mm/yr
  Empirical drate/dGMST = 3.06 mm/yr per deg C

NOTE: This empirical estimate conflates T-driven and t-driven
acceleration. It is an upper bound on the true GMST sensitivity
if any acceleration is due to non-T processes (e.g., ice-sheet dynamics).


---
## 2. Component GMST sensitivities (bottom-up)

Load each component's sensitivity to GMST from saved calibration outputs.

| Component | GMST sensitivity | Source | Notes |
|-----------|-----------------|--------|-------|
| Thermosteric | $b_{\rm th}$, $a_{\rm th}$ | Stage-1 Bayesian fit | Polynomial rate model |
| Greenland SMB | $C_T$ | Literature compilation | Linear in GMST |
| Glaciers | (implicit in trend) | Linear DOLS | T-t collinearity absorbs T signal |
| EAIS | $C_T \approx +60$ Gt/yr/C | Literature | Mass gain (reduces SLR) |
| Peninsula | $C_T \approx -15$ Gt/yr/C | Literature | Negligible |
| WAIS | Insensitive | A4 framework | Driven by ocean circulation, not GMST |
| TWS | Small | Frederikse (obs) / IPCC (proj) | Smaller than thermo + Greenland SMB |

In [4]:
# ── 2a. Thermosteric ──
with open(S1_JSON) as f:
    s1 = json.load(f)

tc = s1['thermosteric_calibration']
a_th = tc['a_th_median_mm']     # mm/yr/C^2  (quadratic)
b_th = tc['b_th_median_mm']     # mm/yr/C    (linear)
c_th = tc['c_th_median_mm']     # mm/yr      (trend)
a_th_ci = tc['a_th_90ci_mm']
b_th_ci = tc['b_th_90ci_mm']

print('Thermosteric (Stage-1 Bayesian fit to Frederikse steric):')
print(f'  rate(T) = {a_th:.3f} T^2 + {b_th:.3f} T + {c_th:.3f}  [mm/yr]')
print(f'  a_th = {a_th:.3f}  [{a_th_ci[0]:.3f}, {a_th_ci[1]:.3f}]')
print(f'  b_th = {b_th:.3f}  [{b_th_ci[0]:.3f}, {b_th_ci[1]:.3f}]')

# ── 2b. Greenland SMB ──
with open(GR_JSON) as f:
    gr = json.load(f)

# Data-driven C_T from GRACE-implied SMB vs GMST
gr_gmst = gr['fits']['GMST']
C_T_gr_data = gr_gmst['C_T_gt_per_C']       # Gt/yr/C (data-driven)
C_T_gr_data_se = gr_gmst['C_T_se_gt_per_C']

# Literature C_T from component_results.h5
with h5py.File(H5_COMP, 'r') as f:
    C_T_gr_lit = -f['greenland/smb_sensitivity'].attrs['C_T']         # Gt/yr/C (sign: positive = mass loss)
    C_T_gr_lit_sigma = f['greenland/smb_sensitivity'].attrs['C_T_sigma']

print(f'\nGreenland SMB sensitivity to GMST:')
print(f'  Data-driven:  C_T = {C_T_gr_data:.0f} +/- {C_T_gr_data_se:.0f} Gt/yr/C'
      f'  ({C_T_gr_data * GT_TO_MM_SLE:.3f} mm/yr SLE per C)')
print(f'  Literature:   C_T = {C_T_gr_lit:.0f} +/- {C_T_gr_lit_sigma:.0f} Gt/yr/C'
      f'  ({C_T_gr_lit * GT_TO_MM_SLE:.3f} mm/yr SLE per C)')

# ── 2c. Glaciers (from DOLS posteriors) ──
with h5py.File(H5_COMP, 'r') as f:
    gl_post = f['glacier/posteriors/posterior_samples'][:]

gl_alpha0 = np.median(gl_post[:, 0]) * M_TO_MM  # m -> mm/yr/C
gl_trend = np.median(gl_post[:, 1]) * M_TO_MM    # m -> mm/yr
print(f'\nGlacier DOLS posteriors:')
print(f'  alpha0 (T sensitivity): {gl_alpha0:.3f} mm/yr/C  (low: T-t collinear in 24-yr record)')
print(f'  trend:                  {gl_trend:.3f} mm/yr  (absorbs most T signal)')

# ── 2d. Other components ──
C_T_eais_gt = 60.0   # Gt/yr/C (mass gain -> negative SLR contribution)
C_T_pen_gt = -15.0    # Gt/yr/C (mass loss -> positive SLR contribution)

print(f'\nOther components (literature):')
print(f'  EAIS:      C_T = +{C_T_eais_gt:.0f} Gt/yr/C  ({C_T_eais_gt * GT_TO_MM_SLE:+.3f} mm/yr SLE per C, reduces SLR)')
print(f'  Peninsula: C_T = {C_T_pen_gt:.0f} Gt/yr/C  ({-C_T_pen_gt * GT_TO_MM_SLE:+.3f} mm/yr SLE per C)')
print(f'  WAIS:      Insensitive to GMST')
print(f'  TWS:       Assumed small relative to thermosteric + Greenland SMB')

Thermosteric (Stage-1 Bayesian fit to Frederikse steric):
  rate(T) = 0.237 T^2 + 1.097 T + 0.990  [mm/yr]
  a_th = 0.237  [0.028, 0.556]
  b_th = 1.097  [0.932, 1.327]

Greenland SMB sensitivity to GMST:
  Data-driven:  C_T = 177 +/- 65 Gt/yr/C  (0.489 mm/yr SLE per C)
  Literature:   C_T = 200 +/- 80 Gt/yr/C  (0.552 mm/yr SLE per C)

Glacier DOLS posteriors:
  alpha0 (T sensitivity): 0.030 mm/yr/C  (low: T-t collinear in 24-yr record)
  trend:                  0.666 mm/yr  (absorbs most T signal)

Other components (literature):
  EAIS:      C_T = +60 Gt/yr/C  (+0.166 mm/yr SLE per C, reduces SLR)
  Peninsula: C_T = -15 Gt/yr/C  (+0.041 mm/yr SLE per C)
  WAIS:      Insensitive to GMST
  TWS:       Assumed small relative to thermosteric + Greenland SMB


In [5]:
# ── Summary: component sensitivities at representative GMST ──
# Evaluate drate/dT = 2*a*T + b at mean satellite-era GMST
T_mean_sat = 0.5 * (T_start + T_end)
T_current = T_end

print(f'Component drate/dGMST evaluated at satellite-era mean T = {T_mean_sat:.2f} C')
print(f'and at current T = {T_current:.2f} C')
print(f'{"":-<72}')

# Use the literature C_T for Greenland SMB (the value used in our projections)
C_T_gr_mm = C_T_gr_lit * GT_TO_MM_SLE  # mm/yr SLE per C

components = {
    'Thermosteric':  {'drate_dT_mean': 2 * a_th * T_mean_sat + b_th,
                      'drate_dT_curr': 2 * a_th * T_current + b_th,
                      'note': f'2*{a_th:.3f}*T + {b_th:.3f}'},
    'Greenland SMB': {'drate_dT_mean': C_T_gr_mm,
                      'drate_dT_curr': C_T_gr_mm,
                      'note': f'C_T = {C_T_gr_lit:.0f} Gt/yr/C (linear)'},
    'Glaciers':      {'drate_dT_mean': gl_alpha0,
                      'drate_dT_curr': gl_alpha0,
                      'note': 'Explicit T term (trend absorbs most)'},
    'EAIS':          {'drate_dT_mean': -C_T_eais_gt * GT_TO_MM_SLE,
                      'drate_dT_curr': -C_T_eais_gt * GT_TO_MM_SLE,
                      'note': 'Mass gain reduces SLR'},
    'Peninsula':     {'drate_dT_mean': -C_T_pen_gt * GT_TO_MM_SLE,
                      'drate_dT_curr': -C_T_pen_gt * GT_TO_MM_SLE,
                      'note': 'Negligible'},
}

total_mean = sum(c['drate_dT_mean'] for c in components.values())
total_curr = sum(c['drate_dT_curr'] for c in components.values())

print(f'{"Component":20s} {"drate/dT (mean T)":>18s} {"drate/dT (curr T)":>18s}   Notes')
print(f'{"":-<100}')
for name, vals in components.items():
    print(f'{name:20s} {vals["drate_dT_mean"]:+18.3f} {vals["drate_dT_curr"]:+18.3f}   {vals["note"]}')
print(f'{"":-<100}')
print(f'{"TOTAL (components)":20s} {total_mean:+18.3f} {total_curr:+18.3f}')
print(f'{"Observed (empirical)":20s} {drate_dGMST_obs:+18.3f} {drate_dGMST_obs:+18.3f}')
print(f'{"":-<100}')
print(f'{"DEFICIT":20s} {drate_dGMST_obs - total_mean:+18.3f} {drate_dGMST_obs - total_curr:+18.3f}')

Component drate/dGMST evaluated at satellite-era mean T = 0.17 C
and at current T = 0.55 C
------------------------------------------------------------------------
Component             drate/dT (mean T)  drate/dT (curr T)   Notes
----------------------------------------------------------------------------------------------------
Thermosteric                     +1.178             +1.361   2*0.237*T + 1.097
Greenland SMB                    +0.552             +0.552   C_T = 200 Gt/yr/C (linear)
Glaciers                         +0.030             +0.030   Explicit T term (trend absorbs most)
EAIS                             -0.166             -0.166   Mass gain reduces SLR
Peninsula                        +0.041             +0.041   Negligible
----------------------------------------------------------------------------------------------------
TOTAL (components)               +1.635             +1.818
Observed (empirical)             +3.061             +3.061
-----------------------------

---
## 3. Satellite-era rate budget

Compare the component-sum rate to the observed GMSL rate during the satellite
era, using loaded projection ensembles. This gives the absolute rate deficit
(complementing the sensitivity deficit above).

In [6]:
# ── Load component projections (hindcast portion covers satellite era) ──
proj_years, all_proj = load_all_projections()

# Use SSP2-4.5 for the satellite-era hindcast (SSP-independent before ~2020)
ssp_ref = 'SSP2-4.5'

# Narrower window for trend: 1993--2020 (before SSP divergence)
trend_mask = (proj_years >= 1993) & (proj_years <= 2020)
yrs_trend = proj_years[trend_mask]

# ── TWS from Frederikse (observational period) ──
# Frederikse harmonized data is in meters; convert to mm for rate
fred_time = df_frederikse.index.map(
    lambda t: t.year + (t.month - 1) / 12 + (t.day - 1) / 365.25
).values
fred_tws = df_frederikse['tws'].values * M_TO_MM  # meters -> mm

# Rebase TWS to BASELINE_YEAR
mask_base = (fred_time >= BASELINE_YEAR - 0.5) & (fred_time <= BASELINE_YEAR + 0.5)
tws_base = np.nanmean(fred_tws[mask_base])
fred_tws_rb = fred_tws - tws_base

# TWS trend during satellite era
fred_sat = (fred_time >= 1993) & (fred_time <= 2020)
p_tws = np.polyfit(fred_time[fred_sat] - 1993, fred_tws_rb[fred_sat], 1)
tws_rate = p_tws[0]  # mm/yr

# ── Component rates from projection ensembles ──
print(f'Component rates during satellite era (SSP-independent), mm/yr')
print(f'Computed as linear trend in level over 1993--2020')
print(f'{"":-<60}')

comp_rates = {}
for comp_name in sorted(all_proj.keys()):
    samples = all_proj[comp_name][ssp_ref]['samples']  # (N_SAMPLES, n_years)
    median_traj = np.median(samples[:, trend_mask], axis=0) * M_TO_MM  # mm
    p = np.polyfit(yrs_trend - yrs_trend[0], median_traj, 1)
    comp_rates[comp_name] = p[0]  # mm/yr
    print(f'  {comp_name:15s}: {p[0]:+.3f} mm/yr')

comp_rates['tws'] = tws_rate
print(f'  {"tws":15s}: {comp_rates["tws"]:+.3f} mm/yr  [Frederikse et al. 2020]')

total_comp_rate = sum(comp_rates.values())

# Observed rate at mid-satellite-era from quadratic fit
t_mid = 0.5 * (1993 + 2020) - sq.t_start
obs_rate_mid = (sq.coefficients[1] + 2 * sq.coefficients[2] * t_mid) * M_TO_MM

print(f'{"":-<60}')
print(f'  {"Component sum":15s}: {total_comp_rate:+.3f} mm/yr')
print(f'  {"Observed (mid-sat)":15s}: {obs_rate_mid:+.3f} mm/yr')
print(f'  {"Deficit":15s}: {obs_rate_mid - total_comp_rate:+.3f} mm/yr')

Component rates during satellite era (SSP-independent), mm/yr
Computed as linear trend in level over 1993--2020
------------------------------------------------------------
  apeninsula     : +0.033 mm/yr
  eais           : -0.029 mm/yr
  glacier        : +0.641 mm/yr
  greenland      : +0.324 mm/yr
  ocean          : +1.219 mm/yr
  wais           : +0.254 mm/yr
  tws            : +0.315 mm/yr  [Frederikse et al. 2020]
------------------------------------------------------------
  Component sum  : +2.756 mm/yr
  Observed (mid-sat): +3.122 mm/yr
  Deficit        : +0.366 mm/yr


---
## 4. Rate-and-state sensitivity as additional constraint

The Bayesian rate-and-state model provides an independent estimate of how GMSL
rate scales with GMST. While the polynomial coefficients are affected by T--t
collinearity over the full 1900--2018 calibration window (inflating the apparent
GMST sensitivity), the model's in-sample rate trajectory is well-calibrated
(R$^2$ = 0.97).

We load the full posterior samples from `bayesian_ratestate_posterior.h5`
(exported by `bayesian_ratestate.ipynb`) to access:
- `posterior/coefficients` — (n_samples, 4): [dalpha_dT, alpha0, trend, d_diseq]
- `posterior/tau` — (n_samples,): relaxation time
- `rate_vs_T/` — equilibrium rate as a function of GMST (200-point grid)
- `sensitivity_vs_T/` — dRate/dT as a function of GMST
- `record_averaged_sensitivity/` — sensitivity averaged over calibration T range

In [7]:
# ── 4a. Rate-and-state posterior samples ──
if os.path.exists(RS_POSTERIOR_H5):
    with h5py.File(RS_POSTERIOR_H5, 'r') as f:
        rs_post = f['posterior/coefficients'][:]  # (n_samples, 4)
        rs_tau = f['posterior/tau'][:]
        rs_param_names = f['posterior'].attrs['param_names']
        print(f'Rate-and-state posterior: {rs_post.shape[0]} samples, '
              f'params: {list(rs_param_names)}')
        for i, name in enumerate(rs_param_names):
            med = np.median(rs_post[:, i]) * M_TO_MM
            p5, p95 = np.percentile(rs_post[:, i] * M_TO_MM, [5, 95])
            print(f'  {name:12s}: {med:+.3f} mm/yr  [{p5:.3f}, {p95:.3f}]')

        # Sensitivity vs T (pre-computed on a grid)
        T_grid = f['sensitivity_vs_T/T_grid_C'][:]
        sens_med = f['sensitivity_vs_T/sensitivity_median_mm_yr_C'][:]
        sens_p5 = f['sensitivity_vs_T/sensitivity_p5_mm_yr_C'][:]
        sens_p95 = f['sensitivity_vs_T/sensitivity_p95_mm_yr_C'][:]

        # Record-averaged sensitivity
        rec_sens_med = float(f['record_averaged_sensitivity/median_mm_yr_C'][()])
        rec_sens_p5 = float(f['record_averaged_sensitivity/p5_mm_yr_C'][()])
        rec_sens_p95 = float(f['record_averaged_sensitivity/p95_mm_yr_C'][()])
        print(f'\n  Record-averaged dRate/dT: {rec_sens_med:.2f} [{rec_sens_p5:.2f}, {rec_sens_p95:.2f}] mm/yr/C')
else:
    print(f'WARNING: {RS_POSTERIOR_H5} not found.')
    print('Run bayesian_ratestate.ipynb to export posterior samples.')
    rs_post = None

# ── 4b. Projection comparison at 2100 ──
rs_npz = np.load('../data/processed/bayesian_ratestate_projections.npz')

print(f'\nProjections at 2100 (mm relative to {BASELINE_YEAR}):')
print(f'{"Source":25s} {"SSP1-2.6":>12s} {"SSP2-4.5":>12s} {"SSP3-7.0":>12s} {"SSP5-8.5":>12s}')
print(f'{"":-<75}')

# Rate-and-state
rs_proj = rs['projections_at_2100_mm']
vals = [f"{rs_proj[s]['median']:.0f}" for s in ['SSP1-2.6', 'SSP2-4.5', 'SSP3-7.0', 'SSP5-8.5']]
print(f'{"Rate-and-state":25s} {vals[0]:>12s} {vals[1]:>12s} {vals[2]:>12s} {vals[3]:>12s}')

# Component sum
idx_2100 = np.argmin(np.abs(proj_years - 2100))
comp_sum_2100 = {}
for ssp in PROJ_SSPS:
    total = np.zeros(N_SAMPLES)
    for comp in all_proj:
        total += all_proj[comp][ssp]['samples'][:, idx_2100]
    comp_sum_2100[ssp] = np.median(total) * M_TO_MM

vals = [f"{comp_sum_2100[s]:.0f}" for s in PROJ_SSPS]
print(f'{"Component sum":25s} {vals[0]:>12s} {vals[1]:>12s} {vals[2]:>12s} {vals[3]:>12s}')

# Satellite-era quadratic (Hamlington methodology)
dt_2100 = 2100 - sq.t_start
sq_2100 = (sq.coefficients[0] + sq.coefficients[1] * dt_2100 + sq.coefficients[2] * dt_2100**2) * M_TO_MM
dt_base = BASELINE_YEAR - sq.t_start
sq_base = sq.coefficients[0] + sq.coefficients[1] * dt_base + sq.coefficients[2] * dt_base**2
sq_2100_rb = sq_2100 - sq_base * M_TO_MM
print(f'{"Sat-era quadratic":25s} {sq_2100_rb:>12.0f} {"(SSP-independent)":>36s}')

Rate-and-state posterior: 320000 samples, params: ['dalpha_dT', 'alpha0', 'trend', 'd_diseq']
  dalpha_dT   : +4.592 mm/yr  [3.276, 5.891]
  alpha0      : +5.202 mm/yr  [4.216, 6.218]
  trend       : +2.626 mm/yr  [2.448, 3.034]
  d_diseq     : +0.499 mm/yr  [0.035, 2.010]

  Record-averaged dRate/dT: 3.56 [2.99, 4.21] mm/yr/C

Projections at 2100 (mm relative to 2005.0):
Source                        SSP1-2.6     SSP2-4.5     SSP3-7.0     SSP5-8.5
---------------------------------------------------------------------------
Rate-and-state                     782         1308         2003         2777
Component sum                      711          826          940         1052
Sat-era quadratic                  615                    (SSP-independent)


---
## 5. Reconciliation framework

### Approach

The observed GMSL rate during the satellite era, minus the well-constrained
components, gives the **residual rate** that must be explained by thermosteric
expansion + Greenland SMB:

$$\dot{H}_{\rm residual}(t) = \dot{H}_{\rm obs}(t) - \dot{H}_{\rm glaciers}(t)
- \dot{H}_{\rm discharge}(t) - \dot{H}_{\rm EAIS}(t) - \dot{H}_{\rm peninsula}(t)
- \dot{H}_{\rm TWS}(t) - \dot{H}_{\rm WAIS}(t)$$

This residual rate, evaluated as a function of GMST, constrains the **sum** of
thermosteric and Greenland SMB sensitivities:

$$\frac{d\dot{H}_{\rm residual}}{dT} = \frac{d\dot{H}_{\rm thermo}}{dT}
+ \frac{d\dot{H}_{\rm GrSMB}}{dT}$$

The individual sensitivities are underdetermined by GMSL alone but can be
partitioned using independent constraints:

- **Thermosteric:** NOAA/EN4 ocean heat content observations
- **Greenland SMB:** RACMO RCM output + literature $C_T$ compilations

### Key property

Unlike the variance-weighted residual allocation in `component_summation.ipynb`,
this approach uses **physics** (GMST sensitivity) rather than statistical
uncertainty to attribute the deficit. The correction scales with projected
temperature rather than decaying with forecast horizon, because the same
underestimated sensitivity persists under future warming.

In [8]:
# ── 5a. Residual rate attributable to thermosteric + Greenland SMB ──
# Well-constrained component rates during satellite era (from Section 3)
constrained_rate = (
    comp_rates['glacier']
    + comp_rates['wais']
    + comp_rates['apeninsula']
    + comp_rates['tws']
)

# Greenland discharge (separate from SMB) — load from component_results.h5
with h5py.File(H5_COMP, 'r') as f:
    # Greenland has separate discharge projections
    gr_dyn_samples = f[f'greenland/projections_discharge/{ssp_ref}/samples'][:]

gr_dyn_median = np.median(gr_dyn_samples[:, trend_mask], axis=0) * M_TO_MM
p_dyn = np.polyfit(yrs_trend - yrs_trend[0], gr_dyn_median, 1)
constrained_rate += p_dyn[0]  # add discharge rate

# EAIS rate from projections
with h5py.File(H5_COMP, 'r') as f:
    eais_samples = f[f'eais/projections/{ssp_ref}/samples'][:]

eais_median = np.median(eais_samples[:, trend_mask], axis=0) * M_TO_MM
p_eais = np.polyfit(yrs_trend - yrs_trend[0], eais_median, 1)
constrained_rate += p_eais[0]

# Residual = observed - constrained
residual_rate = obs_rate_mid - constrained_rate

# What do our current thermosteric + Greenland SMB models give?
# Thermosteric rate
ocean_median = np.median(all_proj['ocean'][ssp_ref]['samples'][:, trend_mask], axis=0) * M_TO_MM
p_ocean = np.polyfit(yrs_trend - yrs_trend[0], ocean_median, 1)

# Greenland SMB rate
with h5py.File(H5_COMP, 'r') as f:
    gr_smb_samples = f[f'greenland/projections_smb/{ssp_ref}/samples'][:]

gr_smb_median = np.median(gr_smb_samples[:, trend_mask], axis=0) * M_TO_MM
p_smb = np.polyfit(yrs_trend - yrs_trend[0], gr_smb_median, 1)

modeled_thermo_smb_rate = p_ocean[0] + p_smb[0]

print(f'Residual rate analysis (1993--2020):')
print(f'  Observed GMSL rate (mid-era):       {obs_rate_mid:.3f} mm/yr')
print(f'  Well-constrained components:        {constrained_rate:.3f} mm/yr')
print(f'    (glaciers + discharge + EAIS + peninsula + TWS + WAIS)')
print(f'  Residual (obs - constrained):       {residual_rate:.3f} mm/yr')
print(f'')
print(f'  Modeled thermosteric rate:           {p_ocean[0]:.3f} mm/yr')
print(f'  Modeled Greenland SMB rate:          {p_smb[0]:.3f} mm/yr')
print(f'  Modeled thermo + SMB:                {modeled_thermo_smb_rate:.3f} mm/yr')
print(f'')
print(f'  Rate deficit (thermo + SMB):         {residual_rate - modeled_thermo_smb_rate:.3f} mm/yr')
print(f'  Deficit as fraction of modeled:      {(residual_rate - modeled_thermo_smb_rate) / modeled_thermo_smb_rate * 100:.1f}%')

Residual rate analysis (1993--2020):
  Observed GMSL rate (mid-era):       3.122 mm/yr
  Well-constrained components:        1.324 mm/yr
    (glaciers + discharge + EAIS + peninsula + TWS + WAIS)
  Residual (obs - constrained):       1.798 mm/yr

  Modeled thermosteric rate:           1.219 mm/yr
  Modeled Greenland SMB rate:          0.213 mm/yr
  Modeled thermo + SMB:                1.432 mm/yr

  Rate deficit (thermo + SMB):         0.366 mm/yr
  Deficit as fraction of modeled:      25.5%


In [9]:
# ── 5b. Partition the deficit between thermosteric and Greenland SMB ──
#
# The deficit in rate (Delta_r) must be absorbed by adjusting the GMST
# sensitivities of thermosteric and/or Greenland SMB.
#
# If we write the adjustment as:
#   Delta_r = delta_b_th * T_mean + delta_C_T_gr * T_mean
#           = (delta_b_th + delta_C_T_gr) * T_mean
#
# Then the total sensitivity adjustment is:
#   delta_total = Delta_r / T_mean
#
# We partition using the ratio of independent uncertainty:
#   w_th = sigma_b_th^2 / (sigma_b_th^2 + sigma_C_T_gr^2)
#   w_gr = 1 - w_th

rate_deficit = residual_rate - modeled_thermo_smb_rate

# Uncertainty in each sensitivity
sigma_b_th = 0.5 * (b_th_ci[1] - b_th_ci[0]) / 1.645  # 90% CI -> 1sigma
sigma_C_T_gr = C_T_gr_lit_sigma * GT_TO_MM_SLE  # mm/yr SLE per C

w_th = sigma_b_th**2 / (sigma_b_th**2 + sigma_C_T_gr**2)
w_gr = 1.0 - w_th

# Sensitivity adjustment (assuming deficit scales with T)
delta_sensitivity_total = rate_deficit / T_mean_sat if abs(T_mean_sat) > 0.01 else np.nan
delta_b_th_adj = w_th * delta_sensitivity_total
delta_C_T_gr_adj = w_gr * delta_sensitivity_total  # mm/yr SLE per C

b_th_adjusted = b_th + delta_b_th_adj
C_T_gr_adjusted_mm = C_T_gr_mm + delta_C_T_gr_adj
C_T_gr_adjusted_gt = C_T_gr_adjusted_mm / GT_TO_MM_SLE

print(f'Deficit partition (variance-weighted by independent uncertainty):')
print(f'  sigma_b_th    = {sigma_b_th:.3f} mm/yr/C')
print(f'  sigma_C_T_gr  = {sigma_C_T_gr:.3f} mm/yr SLE per C')
print(f'  w_th = {w_th:.3f},  w_gr = {w_gr:.3f}')
print(f'')
print(f'  Total sensitivity deficit: {delta_sensitivity_total:+.3f} mm/yr per C')
print(f'')
print(f'  Thermosteric adjustment:   delta_b_th = {delta_b_th_adj:+.3f} mm/yr/C')
print(f'    Original b_th: {b_th:.3f},  Adjusted: {b_th_adjusted:.3f} mm/yr/C')
print(f'')
print(f'  Greenland SMB adjustment:  delta_C_T  = {delta_C_T_gr_adj:+.3f} mm/yr SLE per C')
print(f'    Original C_T:  {C_T_gr_lit:.0f} Gt/yr/C,  Adjusted: {C_T_gr_adjusted_gt:.0f} Gt/yr/C')
print(f'    ({C_T_gr_mm:.3f} -> {C_T_gr_adjusted_mm:.3f} mm/yr SLE per C)')

Deficit partition (variance-weighted by independent uncertainty):
  sigma_b_th    = 0.120 mm/yr/C
  sigma_C_T_gr  = 0.221 mm/yr SLE per C
  w_th = 0.228,  w_gr = 0.772

  Total sensitivity deficit: +2.149 mm/yr per C

  Thermosteric adjustment:   delta_b_th = +0.490 mm/yr/C
    Original b_th: 1.097,  Adjusted: 1.587 mm/yr/C

  Greenland SMB adjustment:  delta_C_T  = +1.659 mm/yr SLE per C
    Original C_T:  200 Gt/yr/C,  Adjusted: 801 Gt/yr/C
    (0.552 -> 2.211 mm/yr SLE per C)


---
## 6. Data gaps and next steps

### Data sources

- **Satellite-era quadratic:** Reference values from Hamlington et al. (2024);
  covariance matrix recomputed using their methodology (not provided in
  the publication).
- **TWS (observational):** Frederikse et al. (2020), column `tws` from
  `slr_processed_data.h5`. For projections, use IPCC AR6 land water storage
  distributions from `ipcc_distributions.h5`.
- **Rate-and-state posteriors:** Full samples in
  `bayesian_ratestate_posterior.h5` (exported from `bayesian_ratestate.ipynb`).

### Next steps

1. **Validate partition:** Check whether the adjusted thermosteric sensitivity
   is consistent with NOAA/EN4 OHC observations independently, and whether
   the adjusted Greenland $C_T$ falls within the literature compilation range.

2. **Joint posterior:** Sample from the joint distribution of $(b_{\rm th},
   C_T^{\rm Gr})$ conditioned on the observed residual rate, preserving the
   negative correlation (if one is higher, the other must be lower).

3. **Projection impact:** Reproject thermosteric and Greenland SMB with adjusted
   sensitivities under each SSP. Quantify the change in total SLR at 2050 and
   2100.

4. **Blending integration:** Feed the adjusted component projections into the
   rate-space blending framework in `component_forecast.ipynb`.

5. **Rate-and-state sensitivity curve:** Use `sensitivity_vs_T/` from the
   posterior H5 to overlay the aggregate dRate/dT against the sum of component
   sensitivities as a function of GMST.